# Project 13 — BROKEN notebook (label switching)

This notebook contains **seeded bugs** centred on the mixture's key pitfall. Run it, read the diagnostics, find each bug, and fix it. Clean reference: `notebook.ipynb`; answer key: `BROKEN_BUGS.md` (don't peek first).

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
RNG = 20240601

In [ ]:
from data.generate_data import generate
data = generate(); y = data['y']

### Model — BUG 1: no ordering constraint on the means.

Without forcing $\mu_0 < \mu_1$, the two components are exchangeable and the chains will label-switch.

In [ ]:
with pm.Model() as model:
    w = pm.Dirichlet('w', a=np.array([2.0, 2.0]))
    # BUG 1: unordered means -> label-switching symmetry intact
    mu = pm.Normal('mu', 0.0, 3.0, shape=2)
    sigma = pm.HalfNormal('sigma', 1.0)
    pm.NormalMixture('y', w=w, mu=mu, sigma=sigma, observed=y)
    # BUG 2: too few tune steps to even adapt around the multimodality
    idata = pm.sample(draws=500, tune=150, chains=4, random_seed=RNG,
                      progressbar=False)

In [ ]:
# R-hat for mu will be far from 1.0 — the smoking gun.
print(az.summary(idata, var_names=['mu','w']))

### BUG 3: trusting the per-label posterior mean of `mu` as if identified.

Averaging a label-switched marginal collapses both states toward the overall mean — a meaningless number reported with false confidence.

In [ ]:
mu_post = idata.posterior['mu'].values.reshape(-1, 2)
# BUG 3: this 'mean of mu[0]' mixes both true states across chains
print('reported mu[0] =', mu_post[:,0].mean(), 'mu[1] =', mu_post[:,1].mean())
fig, ax = plt.subplots(figsize=(6,3.5))
ax.hist(mu_post[:,0], bins=40, alpha=0.6, label='mu[0] (should be one mode!)')
ax.hist(mu_post[:,1], bins=40, alpha=0.6, label='mu[1]')
ax.legend(); ax.set_title('Bimodal per-label marginals = label switching')
plt.tight_layout()